# Step 5 — Gradient-boosted trees + threshold calibration on the C1 feature set

Step 4 picked **C1** (108 intrinsic + 103 traj + 17 Block B + 5 PPR = 233 features) as our best feature set, with RF F1=0.8149.

This notebook tries:
1. **LightGBM** with early stopping on temporal val.
2. **XGBoost** as a sanity comparison.
3. **Threshold calibration** on temporal val (`t ∈ [30, 34]`) — the existing notebooks all use threshold=0.5, leaving F1 on the table.
4. **A small hyperparameter sweep** over GBDT settings.
5. Cache feature matrices to `cache/` for downstream steps.

## Splits
- Train: labelled `t ≤ 29`
- Val (early stopping + threshold tuning): `30 ≤ t ≤ 34`
- Test: `t ≥ 35`

Note: this is more conservative than the RF baselines (which used all `t ≤ 34` for train). To make a fair comparison we also report a LightGBM trained on full `t ≤ 34` (no val) at the val-tuned threshold.

In [1]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report, precision_recall_curve
import lightgbm as lgb
import xgboost as xgb

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
CACHE_DIR        = ROOT / "architectures" / "inductive_tx_classification" / "cross_step_tx_graph" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"repo root: {ROOT}")
print(f"cache:     {CACHE_DIR}")

TRAIN_END    = 34
VAL_START    = 30
N_TIME_STEPS = 49
RANDOM_SEED  = 175
MAX_WALLET_DEGREE = 50
PPR_DAMP        = 0.85
PPR_ITERATIONS  = 30
DECAY_BETA      = 0.2
TRAJ_DECAY      = 0.2
MAX_INCIDENT_PER_SIDE = 32
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
np.random.seed(RANDOM_SEED)

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project
cache:     /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project/architectures/inductive_tx_classification/cross_step_tx_graph/cache


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
print("Loading data...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1:1,2:0,3:-1}).astype(np.int8)
tx_id_to_idx = {tid:i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
all_cols = list(tx_features_df.columns)
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + ["in_txs_degree","out_txs_degree"]
feat_cols_intr = [c for c in all_cols if c not in ("txId","Time step") and c not in GRAPH_COLS]
feat_cols_full = [c for c in all_cols if c not in ("txId","Time step")]
F_INTRINSIC = len(feat_cols_intr)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_intr = tx_features_df[feat_cols_intr].fillna(0).astype(np.float32).values
tx_X_full = tx_features_df[feat_cols_full].fillna(0).astype(np.float32).values
agg_named = ["total_BTC","fees","num_input_addresses","num_output_addresses"]
agg_idxs_full = [feat_cols_full.index(c) for c in agg_named]
total_btc_idx_full = feat_cols_full.index("total_BTC")

addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a:i for i,a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
addr_tx_df["w"]  = addr_tx_df["input_address"].map(wallet_addr_to_idx)
addr_tx_df["tx"] = addr_tx_df["txId"].map(tx_id_to_idx)
addr_tx_df = addr_tx_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
tx_addr_df["w"]  = tx_addr_df["output_address"].map(wallet_addr_to_idx)
tx_addr_df["tx"] = tx_addr_df["txId"].map(tx_id_to_idx)
tx_addr_df = tx_addr_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
incident_in  = defaultdict(list); incident_out = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values): incident_in[int(tx)].append(int(w))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values): incident_out[int(tx)].append(int(w))
wallet_in_txs = defaultdict(list); wallet_out_txs = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values): wallet_in_txs[int(w)].append((int(tx), int(tx_time[tx])))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values): wallet_out_txs[int(w)].append((int(tx), int(tx_time[tx])))
for w in wallet_in_txs:  wallet_in_txs[w].sort(key=lambda r:r[1])
for w in wallet_out_txs: wallet_out_txs[w].sort(key=lambda r:r[1])
all_incidence = defaultdict(set)
for w, lst in wallet_in_txs.items():
    for tx,_ in lst: all_incidence[w].add(tx)
for w, lst in wallet_out_txs.items():
    for tx,_ in lst: all_incidence[w].add(tx)
all_edges_master = pd.concat([
    addr_tx_df[["w","tx"]].assign(t=tx_time[addr_tx_df["tx"].values]),
    tx_addr_df[["w","tx"]].assign(t=tx_time[tx_addr_df["tx"].values]),
], ignore_index=True).drop_duplicates(subset=["w","tx"]).reset_index(drop=True)
print(f"  N_tx={N_tx:,}  F_INTRINSIC={F_INTRINSIC}  bipartite_edges={len(all_edges_master):,}")

Loading data...


  N_tx=203,769  F_INTRINSIC=108  bipartite_edges=1,268,260


In [3]:
# Build (or load from cache) the C1 feature blocks: traj (103) + B (17) + C (5).
TRAJ_NPY = CACHE_DIR / "traj_X.npy"
B_NPY    = CACHE_DIR / "block_B_X.npy"
C_NPY    = CACHE_DIR / "block_C_X.npy"
TRAJ_COLS_PKL = CACHE_DIR / "traj_cols.npy"

BLOCK_B_FEATS = ["B_n_in_edges","B_n_unique_src","B_dt_min","B_dt_mean","B_dt_max","B_decayed_mass",
                 "B_n_illicit_src","B_n_licit_src","B_frac_illicit_src","B_decayed_illicit",
                 "B_src_total_BTC_mean","B_src_total_BTC_max","B_src_fees_mean","B_src_fees_max",
                 "B_src_n_in_addr_mean","B_src_n_out_addr_mean","B_max_src_multiplicity"]
BLOCK_C_FEATS = ["ppr_illicit","ppr_illicit_log","max_nbr_ppr_illicit","mean_nbr_ppr_illicit","n_illicit_seeds_at_t"]
F_B = len(BLOCK_B_FEATS); F_C = len(BLOCK_C_FEATS)

if TRAJ_NPY.exists() and B_NPY.exists() and C_NPY.exists() and TRAJ_COLS_PKL.exists():
    print("Loading cached features...")
    traj_X    = np.load(TRAJ_NPY)
    block_B_X = np.load(B_NPY)
    block_C_X = np.load(C_NPY)
    traj_cols = list(np.load(TRAJ_COLS_PKL, allow_pickle=True))
    F_TRAJ = traj_X.shape[1]
    print(f"  traj_X={traj_X.shape}  block_B_X={block_B_X.shape}  block_C_X={block_C_X.shape}")
else:
    print("Computing features from scratch (~70s)...")
    # ----- Cross-step graph (for Block B) -----
    ROLE_OUT_IN, ROLE_OUT_OUT, ROLE_IN_IN, ROLE_IN_OUT = 0,1,2,3
    src_l, dst_l, dt_l, role_l = [], [], [], []
    for w in all_incidence:
        if len(all_incidence[w]) < 2 or len(all_incidence[w]) > MAX_WALLET_DEGREE: continue
        events = []
        for tx, t in wallet_in_txs.get(w, []):  events.append((t, tx, 'in'))
        for tx, t in wallet_out_txs.get(w, []): events.append((t, tx, 'out'))
        events.sort(key=lambda r: r[0])
        K = len(events)
        for i in range(K):
            ti, txi, si = events[i]
            for j in range(i+1, K):
                tj, txj, sj = events[j]
                if tj <= ti or txi == txj: continue
                if   si=='out' and sj=='in':  role = ROLE_OUT_IN
                elif si=='out' and sj=='out': role = ROLE_OUT_OUT
                elif si=='in'  and sj=='in':  role = ROLE_IN_IN
                else:                          role = ROLE_IN_OUT
                src_l.append(txi); dst_l.append(txj); dt_l.append(tj-ti); role_l.append(role)
    src = np.array(src_l, dtype=np.int64); dst = np.array(dst_l, dtype=np.int64)
    dt_edges = np.array(dt_l, dtype=np.int32); role_e = np.array(role_l, dtype=np.int8)
    order_in = np.argsort(dst, kind="stable")
    dst_in = dst[order_in]; src_in = src[order_in]
    dt_in  = dt_edges[order_in]; role_in = role_e[order_in]
    in_offsets = np.searchsorted(dst_in, np.arange(N_tx + 1)).astype(np.int64)
    print(f"  E_cross_step={len(src):,}")
    # ----- Block B -----
    block_B_X = np.zeros((N_tx, F_B), dtype=np.float32)
    src_total_btc_arr = tx_X_full[:, total_btc_idx_full]
    src_agg_feats_arr = tx_X_full[:, agg_idxs_full]
    for j in range(N_tx):
        a, b = in_offsets[j], in_offsets[j+1]
        if a == b: continue
        s_arr=src_in[a:b]; dt_a=dt_in[a:b]; r_a=role_in[a:b]
        bm = (r_a == ROLE_OUT_IN)
        if not bm.any(): continue
        s_B=s_arr[bm]; dt_B=dt_a[bm]; lab_B=tx_label[s_B]
        dec_B = np.exp(-DECAY_BETA*dt_B.astype(np.float64))
        af_B  = src_agg_feats_arr[s_B]
        uniq_B, mult_B = np.unique(s_B, return_counts=True)
        ill = (lab_B==1); lic = (lab_B==0)
        n_ill=int(ill.sum()); n_lic=int(lic.sum()); n_lab=n_ill+n_lic
        block_B_X[j, 0]  = bm.sum(); block_B_X[j, 1]  = uniq_B.size
        block_B_X[j, 2]  = float(dt_B.min()); block_B_X[j, 3] = float(dt_B.mean())
        block_B_X[j, 4]  = float(dt_B.max()); block_B_X[j, 5] = float(dec_B.sum())
        block_B_X[j, 6]  = n_ill; block_B_X[j, 7] = n_lic
        block_B_X[j, 8]  = (n_ill/n_lab) if n_lab>0 else 0.0
        block_B_X[j, 9]  = float(dec_B[ill].sum()) if n_ill>0 else 0.0
        block_B_X[j,10]  = float(af_B[:,0].mean()); block_B_X[j,11] = float(af_B[:,0].max())
        block_B_X[j,12]  = float(af_B[:,1].mean()); block_B_X[j,13] = float(af_B[:,1].max())
        block_B_X[j,14]  = float(af_B[:,2].mean()); block_B_X[j,15] = float(af_B[:,3].mean())
        block_B_X[j,16]  = int(mult_B.max())
    # ----- Block C: PPR seeded from illicit history -----
    N_total = N_w + N_tx
    W_edges = all_edges_master["w"].values.astype(np.int64)
    T_edges = all_edges_master["tx"].values.astype(np.int64)
    T_times = all_edges_master["t"].values.astype(np.int64)
    edge_order = np.argsort(T_times)
    W_sorted = W_edges[edge_order]; T_sorted = T_edges[edge_order]; Tt_sorted = T_times[edge_order]
    ppr_self = np.zeros(N_tx, dtype=np.float32)
    ppr_max_nbr = np.zeros(N_tx, dtype=np.float32)
    ppr_mean_nbr = np.zeros(N_tx, dtype=np.float32)
    ppr_n_seeds = np.zeros(N_tx, dtype=np.float32)
    incident_in_arr  = {tx: list(set(ws)) for tx, ws in incident_in.items()}
    incident_out_arr = {tx: list(set(ws)) for tx, ws in incident_out.items()}
    for t in range(1, N_TIME_STEPS + 1):
        cut = int(np.searchsorted(Tt_sorted, t, side="right"))
        if cut == 0: continue
        we = W_sorted[:cut]; te = T_sorted[:cut]
        rows = np.concatenate([we, N_w + te]); cols = np.concatenate([N_w + te, we])
        data = np.ones(rows.shape[0], dtype=np.float32)
        A = sp.csr_matrix((data,(rows,cols)), shape=(N_total, N_total))
        deg = np.asarray(A.sum(axis=1)).flatten(); inv_deg = np.zeros_like(deg)
        nz = deg>0; inv_deg[nz] = 1.0/deg[nz]
        illicit_seed_idx = np.where((tx_label==1) & (tx_time<t))[0]
        if illicit_seed_idx.size == 0:
            for tx_idx in np.where(tx_time==t)[0]: ppr_n_seeds[tx_idx] = 0.0
            continue
        s = np.zeros(N_total, dtype=np.float32); s[N_w + illicit_seed_idx] = 1.0/illicit_seed_idx.size
        teleport = (1.0-PPR_DAMP)*s; pr = s.copy()
        for _ in range(PPR_ITERATIONS):
            weighted = pr*inv_deg
            pr = teleport + PPR_DAMP*(A@weighted).astype(np.float32)
        for tx_idx in np.where(tx_time==t)[0]:
            in_w = incident_in_arr.get(int(tx_idx), [])
            out_w = incident_out_arr.get(int(tx_idx), [])
            all_nbr = list(set(in_w)|set(out_w))
            ppr_self[tx_idx] = float(pr[N_w + tx_idx])
            ppr_n_seeds[tx_idx] = float(illicit_seed_idx.size)
            if all_nbr:
                v = pr[all_nbr]
                ppr_max_nbr[tx_idx] = float(v.max())
                ppr_mean_nbr[tx_idx] = float(v.mean())
    ppr_self_log = np.log1p(ppr_self * 1e6).astype(np.float32)
    block_C_X = np.stack([ppr_self, ppr_self_log, ppr_max_nbr, ppr_mean_nbr, ppr_n_seeds], axis=1).astype(np.float32)
    # ----- Block traj (103) — call into shared helpers -----
    all_edges_w = pd.concat([
        addr_tx_df[["w","tx"]].assign(t=tx_time[addr_tx_df["tx"].values], tx_lab=tx_label[addr_tx_df["tx"].values]),
        tx_addr_df[["w","tx"]].assign(t=tx_time[tx_addr_df["tx"].values], tx_lab=tx_label[tx_addr_df["tx"].values]),
    ], ignore_index=True).drop_duplicates(subset=["w","tx"]).sort_values(["w","t"]).reset_index(drop=True)
    g = all_edges_w.groupby("w", sort=False)
    wallet_t_arr={}; wallet_tx_arr={}; wallet_lab_arr={}
    for w, sub in g:
        wallet_t_arr[int(w)]=sub["t"].values.astype(np.int64)
        wallet_tx_arr[int(w)]=sub["tx"].values.astype(np.int64)
        wallet_lab_arr[int(w)]=sub["tx_lab"].values.astype(np.int64)
    wallet_event_count = {w:len(t) for w,t in wallet_t_arr.items()}
    wallet_has_illicit_by = {}
    for w, labs in wallet_lab_arr.items():
        m = (labs==1)
        if m.any(): wallet_has_illicit_by[w] = int(wallet_t_arr[w][m].min())
    def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
        if len(wlist) <= k: return list(wlist)
        cnts = np.array([wallet_event_count.get(w,0) for w in wlist], dtype=np.int64)
        order = np.argsort(-cnts, kind="stable")
        return [wlist[i] for i in order[:k]]
    def per_wallet_priors(w, t_query):
        tarr = wallet_t_arr.get(w)
        if tarr is None: return None
        cut = int(np.searchsorted(tarr, t_query, side="left"))
        if cut == 0: return None
        prior_t=tarr[:cut]; prior_lab=wallet_lab_arr[w][:cut]; prior_tx=wallet_tx_arr[w][:cut]
        illicit_mask = (prior_lab==1)
        n_illicit=int(illicit_mask.sum()); n_licit=int((prior_lab==0).sum())
        last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit>0 else -1
        if n_illicit > 0:
            decay_w = np.exp(-TRAJ_DECAY*(t_query-prior_t[illicit_mask]).astype(np.float64))
            decayed_illicit_score = float(decay_w.sum())
        else: decayed_illicit_score = 0.0
        illicit_frac = n_illicit/max(n_illicit+n_licit,1)
        co_wallets = set()
        for tx_i in prior_tx:
            for cw in incident_in.get(int(tx_i),[]):
                if cw != w: co_wallets.add(cw)
            for cw in incident_out.get(int(tx_i),[]):
                if cw != w: co_wallets.add(cw)
            if len(co_wallets) >= MAX_CO_WALLETS: break
        n_co_wallets = len(co_wallets)
        n_co_illicit = sum(1 for cw in co_wallets if wallet_has_illicit_by.get(cw, N_TIME_STEPS+1) < t_query)
        uin, uout = set(), set()
        for tx_i in prior_tx:
            for cw in incident_in.get(int(tx_i),[]):
                if cw != w: uin.add(cw)
            for cw in incident_out.get(int(tx_i),[]):
                if cw != w: uout.add(cw)
        n_inp=len(uin); n_outp=len(uout); fan_asym=n_outp/max(n_inp,1)
        age=int(t_query-prior_t[0]); recency=int(t_query-prior_t[-1])
        n_recent_5=int(((t_query-prior_t)<=5).sum()); n_recent_1=int(((t_query-prior_t)<=1).sum())
        if cut>=2:
            iat = np.diff(prior_t.astype(np.float64))
            iat_mean=float(iat.mean()); iat_std=float(iat.std())
            burstiness = float((iat_std-iat_mean)/(iat_std+iat_mean+1e-8))
        else: iat_mean=0.0; iat_std=0.0; burstiness=0.0
        velocity = n_recent_5/max(cut,1)
        feat_vals = tx_X_full[prior_tx][:, agg_idxs_full]
        return {"n":int(cut),"n_illicit":n_illicit,"n_licit":n_licit,"illicit_frac":illicit_frac,
                "last_illicit_t":last_illicit_t,"decayed_illicit":decayed_illicit_score,
                "n_co_wallets":n_co_wallets,"n_co_illicit":n_co_illicit,
                "co_illicit_frac":n_co_illicit/max(n_co_wallets,1),
                "n_in_partners":n_inp,"n_out_partners":n_outp,"fan_asymmetry":fan_asym,
                "first_seen_t":int(prior_t[0]),"last_seen_t":int(prior_t[-1]),
                "age":age,"recency":recency,"n_recent_5":n_recent_5,"n_recent_1":n_recent_1,
                "iat_mean":iat_mean,"iat_std":iat_std,"burstiness":burstiness,"velocity":velocity,
                "feat_means":feat_vals.mean(axis=0),"feat_maxes":feat_vals.max(axis=0)}
    def aggregate_side(summaries, p, t_T):
        n_total=len(summaries); valid=[s for s in summaries if s is not None]; n_w_prior=len(valid)
        out={f"{p}_n_wallets":n_total, f"{p}_n_wallets_with_prior":n_w_prior,
             f"{p}_frac_first_appearance":(n_total-n_w_prior)/max(n_total,1)}
        keys_zero=["n_priors_sum","n_priors_max","n_illicit_sum","n_illicit_max","n_licit_sum",
                   "n_wallets_with_illicit","n_wallets_illicit_frac_gt0","n_wallets_illicit_frac_gt50",
                   "illicit_frac_max","illicit_frac_mean","decayed_illicit_max","decayed_illicit_sum",
                   "recency_to_illicit_min","co_illicit_sum","co_illicit_max","co_illicit_frac_max",
                   "co_illicit_frac_mean","n_co_wallets_sum","fan_asymmetry_max","fan_asymmetry_mean",
                   "n_in_partners_max","n_out_partners_max","frac_single_use","age_max","age_mean",
                   "recency_min","n_recent_5_sum","n_recent_5_max","n_recent_1_sum","velocity_max",
                   "velocity_mean","burstiness_max","burstiness_mean","iat_mean_min","iat_std_max"]
        for k in keys_zero: out[f"{p}_{k}"]=0.0
        out[f"{p}_recency_to_illicit_min"]=float(RECENCY_SENTINEL); out[f"{p}_recency_min"]=float(RECENCY_SENTINEL)
        for nm in agg_named:
            out[f"{p}_prior_{nm}_mean_max"]=0.0; out[f"{p}_prior_{nm}_max_max"]=0.0
        if not valid: return out
        ns=np.array([s["n"] for s in valid],dtype=np.float64)
        nis=np.array([s["n_illicit"] for s in valid],dtype=np.float64)
        nls=np.array([s["n_licit"] for s in valid],dtype=np.float64)
        ill_frac=np.array([s["illicit_frac"] for s in valid],dtype=np.float64)
        dec_ill=np.array([s["decayed_illicit"] for s in valid],dtype=np.float64)
        last_ill=np.array([s["last_illicit_t"] for s in valid],dtype=np.int64)
        has_ill=(last_ill>=0); rec_ill=np.where(has_ill, t_T-last_ill, RECENCY_SENTINEL).astype(np.float64)
        co_ill=np.array([s["n_co_illicit"] for s in valid],dtype=np.float64)
        co_n=np.array([s["n_co_wallets"] for s in valid],dtype=np.float64)
        co_fr=np.array([s["co_illicit_frac"] for s in valid],dtype=np.float64)
        fan_a=np.array([s["fan_asymmetry"] for s in valid],dtype=np.float64)
        n_inp=np.array([s["n_in_partners"] for s in valid],dtype=np.float64)
        n_outp=np.array([s["n_out_partners"] for s in valid],dtype=np.float64)
        ages=np.array([s["age"] for s in valid],dtype=np.float64)
        recs=np.array([s["recency"] for s in valid],dtype=np.float64)
        nr5=np.array([s["n_recent_5"] for s in valid],dtype=np.float64)
        nr1=np.array([s["n_recent_1"] for s in valid],dtype=np.float64)
        vel=np.array([s["velocity"] for s in valid],dtype=np.float64)
        bur=np.array([s["burstiness"] for s in valid],dtype=np.float64)
        iam=np.array([s["iat_mean"] for s in valid],dtype=np.float64)
        ias=np.array([s["iat_std"] for s in valid],dtype=np.float64)
        fm=np.stack([s["feat_means"] for s in valid],axis=0)
        fx=np.stack([s["feat_maxes"] for s in valid],axis=0)
        out[f"{p}_n_priors_sum"]=float(ns.sum()); out[f"{p}_n_priors_max"]=float(ns.max())
        out[f"{p}_n_illicit_sum"]=float(nis.sum()); out[f"{p}_n_illicit_max"]=float(nis.max())
        out[f"{p}_n_licit_sum"]=float(nls.sum())
        out[f"{p}_n_wallets_with_illicit"]=float(has_ill.sum())
        out[f"{p}_n_wallets_illicit_frac_gt0"]=float((ill_frac>0).sum())
        out[f"{p}_n_wallets_illicit_frac_gt50"]=float((ill_frac>0.5).sum())
        out[f"{p}_illicit_frac_max"]=float(ill_frac.max()); out[f"{p}_illicit_frac_mean"]=float(ill_frac.mean())
        out[f"{p}_decayed_illicit_max"]=float(dec_ill.max()); out[f"{p}_decayed_illicit_sum"]=float(dec_ill.sum())
        out[f"{p}_recency_to_illicit_min"]=float(rec_ill.min())
        out[f"{p}_co_illicit_sum"]=float(co_ill.sum()); out[f"{p}_co_illicit_max"]=float(co_ill.max())
        out[f"{p}_co_illicit_frac_max"]=float(co_fr.max()); out[f"{p}_co_illicit_frac_mean"]=float(co_fr.mean())
        out[f"{p}_n_co_wallets_sum"]=float(co_n.sum())
        out[f"{p}_fan_asymmetry_max"]=float(fan_a.max()); out[f"{p}_fan_asymmetry_mean"]=float(fan_a.mean())
        out[f"{p}_n_in_partners_max"]=float(n_inp.max()); out[f"{p}_n_out_partners_max"]=float(n_outp.max())
        out[f"{p}_frac_single_use"]=sum(1 for s in valid if s["n"]==1)/max(n_w_prior,1)
        out[f"{p}_age_max"]=float(ages.max()); out[f"{p}_age_mean"]=float(ages.mean())
        out[f"{p}_recency_min"]=float(recs.min())
        out[f"{p}_n_recent_5_sum"]=float(nr5.sum()); out[f"{p}_n_recent_5_max"]=float(nr5.max())
        out[f"{p}_n_recent_1_sum"]=float(nr1.sum())
        out[f"{p}_velocity_max"]=float(vel.max()); out[f"{p}_velocity_mean"]=float(vel.mean())
        out[f"{p}_burstiness_max"]=float(bur.max()); out[f"{p}_burstiness_mean"]=float(bur.mean())
        out[f"{p}_iat_mean_min"]=float(iam.min()); out[f"{p}_iat_std_max"]=float(ias.max())
        for k, nm in enumerate(agg_named):
            out[f"{p}_prior_{nm}_mean_max"]=float(fm[:,k].max())
            out[f"{p}_prior_{nm}_max_max"]=float(fx[:,k].max())
        return out
    print("  computing 103 traj features (~40s)...")
    t0=time.time(); traj_rows=[]
    for tx_idx in range(N_tx):
        t_T=int(tx_time[tx_idx])
        in_w  = pick_top_wallets(incident_in.get(tx_idx,[]))
        out_w = pick_top_wallets(incident_out.get(tx_idx,[]))
        in_summ=[per_wallet_priors(w,t_T) for w in in_w]
        out_summ=[per_wallet_priors(w,t_T) for w in out_w]
        row={}; row.update(aggregate_side(in_summ,"in",t_T)); row.update(aggregate_side(out_summ,"out",t_T))
        row["both_sides_have_illicit"]=float(row["in_n_wallets_with_illicit"]>0 and row["out_n_wallets_with_illicit"]>0)
        row["total_n_illicit_priors"]=row["in_n_illicit_sum"]+row["out_n_illicit_sum"]
        row["total_n_wallets_with_illicit"]=row["in_n_wallets_with_illicit"]+row["out_n_wallets_with_illicit"]
        row["total_co_illicit"]=row["in_co_illicit_sum"]+row["out_co_illicit_sum"]
        row["min_recency_to_illicit"]=min(row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"])
        row["max_illicit_frac_either_side"]=max(row["in_illicit_frac_max"], row["out_illicit_frac_max"])
        row["max_decayed_illicit_either"]=max(row["in_decayed_illicit_max"], row["out_decayed_illicit_max"])
        row["max_co_illicit_frac_either"]=max(row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"])
        row["total_frac_first_appearance"]=(
            (row["in_frac_first_appearance"]*max(row["in_n_wallets"],1)+
             row["out_frac_first_appearance"]*max(row["out_n_wallets"],1))/
            max(row["in_n_wallets"]+row["out_n_wallets"],1))
        T_btc=float(tx_X_full[tx_idx,total_btc_idx_full])
        max_p=max(row["in_prior_total_BTC_max_max"], row["out_prior_total_BTC_max_max"])
        mean_p=max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
        row["T_btc_vs_max_prior"]=T_btc/max(max_p,1.0); row["T_btc_vs_mean_prior"]=T_btc/max(mean_p,1.0)
        traj_rows.append(row)
    traj_df=pd.DataFrame(traj_rows); traj_X=traj_df.values.astype(np.float32)
    traj_cols=list(traj_df.columns); F_TRAJ=traj_X.shape[1]
    print(f"  traj done in {time.time()-t0:.0f}s")
    # Save
    np.save(TRAJ_NPY, traj_X)
    np.save(B_NPY, block_B_X)
    np.save(C_NPY, block_C_X)
    np.save(TRAJ_COLS_PKL, np.array(traj_cols, dtype=object))
    print(f"  cached features to {CACHE_DIR}")
F_TRAJ = traj_X.shape[1]
print(f"  feature blocks: intrinsic {F_INTRINSIC}, traj {F_TRAJ}, B {F_B}, C {F_C}")
all_feat_names = list(feat_cols_intr) + list(traj_cols) + BLOCK_B_FEATS + BLOCK_C_FEATS

Computing features from scratch (~70s)...


  E_cross_step=445,180


  computing 103 traj features (~40s)...


  traj done in 43s
  cached features to /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project/architectures/inductive_tx_classification/cross_step_tx_graph/cache
  feature blocks: intrinsic 108, traj 103, B 17, C 5


In [4]:
# C1 feature matrix
X_C1 = np.concatenate([tx_X_intr, traj_X, block_B_X, block_C_X], axis=1)
print(f"X_C1: {X_C1.shape}")

# Three split modes:
#   * for GBDT with early stopping: train on t<=29, val on t in [30,34]
#   * for fair comparison vs RF baselines: train on t<=34 (no val), evaluate on test
labelled = (tx_label != -1)
tr_es_mask    = labelled & (tx_time <= 29)
val_mask      = labelled & (tx_time >= VAL_START) & (tx_time <= TRAIN_END)
tr_full_mask  = labelled & (tx_time <= TRAIN_END)
test_mask     = labelled & (tx_time >  TRAIN_END)

y_tr_es   = tx_label[tr_es_mask]
y_val     = tx_label[val_mask]
y_tr_full = tx_label[tr_full_mask]
y_test    = tx_label[test_mask]
test_t_arr = tx_time[test_mask]

print(f"  tr_es:   n={int(tr_es_mask.sum()):,}  illicit_rate={y_tr_es.mean():.4f}")
print(f"  val:     n={int(val_mask.sum()):,}  illicit_rate={y_val.mean():.4f}")
print(f"  tr_full: n={int(tr_full_mask.sum()):,}  illicit_rate={y_tr_full.mean():.4f}")
print(f"  test:    n={int(test_mask.sum()):,}  illicit_rate={y_test.mean():.4f}")

X_tr_es   = X_C1[tr_es_mask]
X_val     = X_C1[val_mask]
X_tr_full = X_C1[tr_full_mask]
X_test    = X_C1[test_mask]

X_C1: (203769, 233)
  tr_es:   n=26,381  illicit_rate=0.1088
  val:     n=3,513  illicit_rate=0.1682
  tr_full: n=29,894  illicit_rate=0.1158
  test:    n=16,670  illicit_rate=0.0650


In [5]:
# Baseline: RF on C1, full train (matches step4 C1 result).
print("=== RF C1 (matches step4) ===")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                            n_jobs=-1, random_state=RANDOM_SEED)
rf.fit(X_tr_full, y_tr_full)
yp_rf = rf.predict(X_test); ypr_rf = rf.predict_proba(X_test)[:,1]
f1_rf = f1_score(y_test, yp_rf, pos_label=1, zero_division=0)
auc_rf = roc_auc_score(y_test, ypr_rf); pr_rf = average_precision_score(y_test, ypr_rf)
print(f"  F1={f1_rf:.4f}  AUC={auc_rf:.4f}  PR-AUC={pr_rf:.4f}  ({time.time()-t0:.1f}s)")

# Threshold tune on val set
ypr_rf_val = rf.predict_proba(X_val)[:,1]
prec, rec, thr = precision_recall_curve(y_val, ypr_rf_val)
f1s = 2*prec*rec/(prec+rec+1e-12)
if len(thr) > 0:
    bi = int(np.argmax(f1s[:-1]))
    rf_best_thr = float(thr[bi]); rf_best_f1_val = float(f1s[bi])
else:
    rf_best_thr = 0.5; rf_best_f1_val = float(f1s.max())
yp_rf_thr = (ypr_rf >= rf_best_thr).astype(int)
f1_rf_thr = f1_score(y_test, yp_rf_thr, pos_label=1, zero_division=0)
print(f"  val-tuned thr={rf_best_thr:.3f}  val_F1={rf_best_f1_val:.4f}  test_F1={f1_rf_thr:.4f}")

=== RF C1 (matches step4) ===


  F1=0.8149  AUC=0.9209  PR-AUC=0.8067  (2.9s)


  val-tuned thr=0.635  val_F1=1.0000  test_F1=0.7848


In [6]:
# LightGBM small grid search using val for early stopping.
pos_w = (1.0 - y_tr_es.mean()) / y_tr_es.mean()
print(f"  scale_pos_weight = {pos_w:.2f}")

lgb_grid = []
for n_leaves in [31, 63, 127]:
    for lr in [0.05, 0.1]:
        for mcs in [10, 20]:
            lgb_grid.append(dict(num_leaves=n_leaves, learning_rate=lr, min_child_samples=mcs))

results = []
best_lgb = None; best_lgb_pr_val = -1
for params in lgb_grid:
    p = dict(
        objective="binary", metric="average_precision",
        n_estimators=2000, learning_rate=params["learning_rate"],
        num_leaves=params["num_leaves"], min_child_samples=params["min_child_samples"],
        scale_pos_weight=pos_w, verbose=-1, n_jobs=-1, random_state=RANDOM_SEED,
        subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
    )
    m = lgb.LGBMClassifier(**p)
    m.fit(X_tr_es, y_tr_es,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    p_val = m.predict_proba(X_val)[:,1]
    pr_val = average_precision_score(y_val, p_val)
    p_test = m.predict_proba(X_test)[:,1]
    pr_test = average_precision_score(y_test, p_test)
    auc_test = roc_auc_score(y_test, p_test)
    f1_test_05 = f1_score(y_test, (p_test>=0.5).astype(int), pos_label=1, zero_division=0)
    results.append((params, m.best_iteration_, pr_val, pr_test, auc_test, f1_test_05, p_test, p_val))
    print(f"  L={params['num_leaves']:>3d}  lr={params['learning_rate']}  mcs={params['min_child_samples']:>2d}  "
          f"best_iter={m.best_iteration_:>4d}  val_PR={pr_val:.4f}  test_PR={pr_test:.4f}  test_F1@0.5={f1_test_05:.4f}")
    if pr_val > best_lgb_pr_val:
        best_lgb_pr_val = pr_val; best_lgb = (params, m, p_val, p_test, m.best_iteration_)

params, m_lgb, p_val_lgb, p_test_lgb, lgb_best_iter = best_lgb
print(f"\n  selected by val PR-AUC: leaves={params['num_leaves']}  lr={params['learning_rate']}  mcs={params['min_child_samples']}  best_iter={lgb_best_iter}")

  scale_pos_weight = 8.19


  L= 31  lr=0.05  mcs=10  best_iter=   6  val_PR=0.9644  test_PR=0.7259  test_F1@0.5=0.0000


  L= 31  lr=0.05  mcs=20  best_iter=   6  val_PR=0.9756  test_PR=0.7315  test_F1@0.5=0.0000


  L= 31  lr=0.1  mcs=10  best_iter=  77  val_PR=0.9915  test_PR=0.8079  test_F1@0.5=0.7335


  L= 31  lr=0.1  mcs=20  best_iter= 117  val_PR=0.9918  test_PR=0.8024  test_F1@0.5=0.7701


...

  L= 63  lr=0.1  mcs=20  best_iter=  82  val_PR=0.9922  test_PR=0.8000  test_F1@0.5=0.7635


  L=127  lr=0.05  mcs=10  best_iter=   5  val_PR=0.9719  test_PR=0.6715  test_F1@0.5=0.0000


  L=127  lr=0.05  mcs=20  best_iter=   5  val_PR=0.9710  test_PR=0.7036  test_F1@0.5=0.0000


  L=127  lr=0.1  mcs=10  best_iter= 419  val_PR=0.9904  test_PR=0.7979  test_F1@0.5=0.8033


  L=127  lr=0.1  mcs=20  best_iter= 322  val_PR=0.9910  test_PR=0.8005  test_F1@0.5=0.7958

  selected by val PR-AUC: leaves=63  lr=0.1  mcs=20  best_iter=82


In [7]:
# Refit LightGBM on full train (t<=34) for fair comparison vs RF baselines.
p_full = dict(
    objective="binary",
    n_estimators=lgb_best_iter,    # use the val-selected number of trees
    learning_rate=params["learning_rate"],
    num_leaves=params["num_leaves"],
    min_child_samples=params["min_child_samples"],
    scale_pos_weight=(1.0-y_tr_full.mean())/y_tr_full.mean(),
    verbose=-1, n_jobs=-1, random_state=RANDOM_SEED,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
)
m_lgb_full = lgb.LGBMClassifier(**p_full)
m_lgb_full.fit(X_tr_full, y_tr_full)
p_test_lgb_full = m_lgb_full.predict_proba(X_test)[:,1]
p_val_lgb_full  = m_lgb_full.predict_proba(X_val)[:,1]   # for threshold calibration

# Threshold calibration on val
prec, rec, thr = precision_recall_curve(y_val, p_val_lgb_full)
f1s = 2*prec*rec/(prec+rec+1e-12)
if len(thr) > 0:
    bi = int(np.argmax(f1s[:-1])); lgb_thr = float(thr[bi]); lgb_val_f1 = float(f1s[bi])
else:
    lgb_thr = 0.5; lgb_val_f1 = float(f1s.max())

yp_lgb_05  = (p_test_lgb_full >= 0.5).astype(int)
yp_lgb_thr = (p_test_lgb_full >= lgb_thr).astype(int)
f1_lgb_05 = f1_score(y_test, yp_lgb_05, zero_division=0)
f1_lgb_thr = f1_score(y_test, yp_lgb_thr, zero_division=0)
auc_lgb = roc_auc_score(y_test, p_test_lgb_full)
pr_lgb  = average_precision_score(y_test, p_test_lgb_full)
print(f"=== LightGBM (full train t<=34, hyperparams from val) ===")
print(f"  F1@0.5             = {f1_lgb_05:.4f}")
print(f"  F1@val-tuned thr   = {f1_lgb_thr:.4f}  (thr={lgb_thr:.3f}, val_F1={lgb_val_f1:.4f})")
print(f"  AUC                = {auc_lgb:.4f}")
print(f"  PR-AUC             = {pr_lgb:.4f}")
print(classification_report(y_test, yp_lgb_thr, target_names=["licit","illicit"], digits=4, zero_division=0))

=== LightGBM (full train t<=34, hyperparams from val) ===
  F1@0.5             = 0.8004
  F1@val-tuned thr   = 0.7842  (thr=0.952, val_F1=1.0000)
  AUC                = 0.9279
  PR-AUC             = 0.8041
              precision    recall  f1-score   support

       licit     0.9762    0.9996    0.9877     15587
     illicit     0.9901    0.6491    0.7842      1083

    accuracy                         0.9768     16670
   macro avg     0.9832    0.8243    0.8859     16670
weighted avg     0.9771    0.9768    0.9745     16670



In [8]:
# XGBoost comparison (sanity)
import xgboost as xgb
pos_w_full = (1.0-y_tr_full.mean())/y_tr_full.mean()

# Use val for early stopping
m_xgb = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=6,
    subsample=0.85, colsample_bytree=0.85,
    scale_pos_weight=pos_w_full, eval_metric="aucpr",
    n_jobs=-1, random_state=RANDOM_SEED, early_stopping_rounds=50, verbosity=0,
)
m_xgb.fit(X_tr_es, y_tr_es, eval_set=[(X_val, y_val)], verbose=False)
best_iter_xgb = m_xgb.best_iteration
print(f"  xgb best_iter (early stop on val): {best_iter_xgb}")

# Refit on full train at that iter count
m_xgb_full = xgb.XGBClassifier(
    n_estimators=best_iter_xgb+1, learning_rate=0.05, max_depth=6,
    subsample=0.85, colsample_bytree=0.85,
    scale_pos_weight=pos_w_full, eval_metric="aucpr",
    n_jobs=-1, random_state=RANDOM_SEED, verbosity=0,
)
m_xgb_full.fit(X_tr_full, y_tr_full)
p_test_xgb = m_xgb_full.predict_proba(X_test)[:,1]
p_val_xgb  = m_xgb_full.predict_proba(X_val)[:,1]
prec, rec, thr = precision_recall_curve(y_val, p_val_xgb)
f1s = 2*prec*rec/(prec+rec+1e-12)
if len(thr) > 0:
    bi = int(np.argmax(f1s[:-1])); xgb_thr = float(thr[bi])
else: xgb_thr = 0.5
yp_xgb_05  = (p_test_xgb >= 0.5).astype(int)
yp_xgb_thr = (p_test_xgb >= xgb_thr).astype(int)
f1_xgb_05 = f1_score(y_test, yp_xgb_05, zero_division=0)
f1_xgb_thr = f1_score(y_test, yp_xgb_thr, zero_division=0)
auc_xgb = roc_auc_score(y_test, p_test_xgb)
pr_xgb = average_precision_score(y_test, p_test_xgb)
print(f"=== XGBoost (full train t<=34, hyperparams from val) ===")
print(f"  F1@0.5             = {f1_xgb_05:.4f}")
print(f"  F1@val-tuned thr   = {f1_xgb_thr:.4f}  (thr={xgb_thr:.3f})")
print(f"  AUC                = {auc_xgb:.4f}")
print(f"  PR-AUC             = {pr_xgb:.4f}")

  xgb best_iter (early stop on val): 29


=== XGBoost (full train t<=34, hyperparams from val) ===
  F1@0.5             = 0.6264
  F1@val-tuned thr   = 0.7845  (thr=0.669)
  AUC                = 0.9266
  PR-AUC             = 0.8034


In [9]:
# Soft-vote ensemble of RF + LightGBM + XGBoost with weights tuned on val PR-AUC.
# Quick grid over (w_rf, w_lgb, w_xgb) on simplex.
p_val_rf  = rf.predict_proba(X_val)[:,1]
p_val_lgb = p_val_lgb_full
p_val_xgb = p_val_xgb

best_w = None; best_pr_val = -1
weights = []
for a in np.arange(0, 1.01, 0.1):
    for b in np.arange(0, 1.01-a, 0.1):
        c = 1.0 - a - b
        if c < -1e-9: continue
        if c < 0: c = 0
        weights.append((a, b, c))
for (a,b,c) in weights:
    p = a*p_val_rf + b*p_val_lgb + c*p_val_xgb
    pr = average_precision_score(y_val, p)
    if pr > best_pr_val:
        best_pr_val = pr; best_w = (a,b,c)
print(f"  best ensemble weights (RF, LGB, XGB) = {best_w}  val_PR={best_pr_val:.4f}")

p_test_rf  = ypr_rf
p_test_ens = best_w[0]*p_test_rf + best_w[1]*p_test_lgb_full + best_w[2]*p_test_xgb
p_val_ens  = best_w[0]*p_val_rf  + best_w[1]*p_val_lgb_full  + best_w[2]*p_val_xgb
prec, rec, thr = precision_recall_curve(y_val, p_val_ens)
f1s = 2*prec*rec/(prec+rec+1e-12)
ens_thr = float(thr[int(np.argmax(f1s[:-1]))]) if len(thr)>0 else 0.5
yp_ens_thr = (p_test_ens >= ens_thr).astype(int)
yp_ens_05  = (p_test_ens >= 0.5).astype(int)
f1_ens_05 = f1_score(y_test, yp_ens_05, zero_division=0)
f1_ens_thr = f1_score(y_test, yp_ens_thr, zero_division=0)
auc_ens = roc_auc_score(y_test, p_test_ens); pr_ens = average_precision_score(y_test, p_test_ens)
print(f"=== Ensemble (RF+LGB+XGB, val-tuned weights) ===")
print(f"  F1@0.5={f1_ens_05:.4f}  F1@val-thr={f1_ens_thr:.4f}  (thr={ens_thr:.3f})  AUC={auc_ens:.4f}  PR-AUC={pr_ens:.4f}")

  best ensemble weights (RF, LGB, XGB) = (np.float64(0.0), np.float64(0.6000000000000001), np.float64(0.3999999999999999))  val_PR=1.0000
=== Ensemble (RF+LGB+XGB, val-tuned weights) ===
  F1@0.5=0.8062  F1@val-thr=0.8118  (thr=0.660)  AUC=0.9274  PR-AUC=0.8073


In [10]:
print("="*78)
print(f"{'Model':50s}  {'F1@0.5':>7s}  {'F1@thr':>7s}  {'AUC':>7s}  {'PR-AUC':>7s}")
print("="*78)
rows = [
    ("RF C1 [233]",                       f1_rf,    f1_rf_thr,  auc_rf,  pr_rf),
    ("LightGBM C1 [233]",                 f1_lgb_05, f1_lgb_thr, auc_lgb, pr_lgb),
    ("XGBoost  C1 [233]",                 f1_xgb_05, f1_xgb_thr, auc_xgb, pr_xgb),
    ("Ensemble RF+LGB+XGB (val weights)", f1_ens_05, f1_ens_thr, auc_ens, pr_ens),
]
for n,f1_05,f1_th,auc,pr in rows:
    print(f"  {n:48s}  {f1_05:>7.4f}  {f1_th:>7.4f}  {auc:>7.4f}  {pr:>7.4f}")
print()
print("  Reference baselines (from earlier steps):")
print(f"    A0 RF [108 intrinsic, no graph]       F1=0.8021  AUC=0.9026  PR-AUC=0.7855")
print(f"    A1 RF [108 + 103 traj]                F1=0.8098  AUC=0.9196  PR-AUC=0.8029")
print(f"    A3 RF [A1 + 17 Block B]               F1=0.8122  AUC=0.9160  PR-AUC=0.8027")
print(f"    C1 RF [A3 + 5 PPR] (step 4)           F1=0.8149  AUC=0.9209  PR-AUC=0.8067")

# Per-timestep F1 — best model only
print("\n"+"="*78)
print("Per-timestep test F1 — best model (val-tuned threshold)")
print("="*78)
best_name = max(rows, key=lambda r: r[2])[0]
best_yp = {
    "RF C1 [233]":                       (ypr_rf, rf_best_thr),
    "LightGBM C1 [233]":                 (p_test_lgb_full, lgb_thr),
    "XGBoost  C1 [233]":                 (p_test_xgb, xgb_thr),
    "Ensemble RF+LGB+XGB (val weights)": (p_test_ens, ens_thr),
}[best_name]
best_p, best_thr = best_yp
yp_t = (best_p >= best_thr).astype(int)
print(f"best by F1@thr: {best_name}")
for t in range(TRAIN_END+1, N_TIME_STEPS+1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    f1t = f1_score(yt, yp_t[m], pos_label=1, zero_division=0) if yt.sum()>0 else float('nan')
    print(f"  t={t:>2d}  n={int(m.sum()):>4d}  illicit={int(yt.sum()):>3d}  F1={f1t:.4f}")

# Top-30 importance from LightGBM
print("\n"+"="*78)
print("Top-30 LightGBM features (gain importance)")
print("="*78)
imp = m_lgb_full.booster_.feature_importance(importance_type="gain")
order = np.argsort(-imp)[:30]
F_off_traj = F_INTRINSIC + F_TRAJ; F_off_B = F_off_traj + F_B; F_off_C = F_off_B + F_C
for rank, i in enumerate(order, 1):
    if i < F_INTRINSIC:    tag = ""
    elif i < F_off_traj:   tag = "  (TRAJ)"
    elif i < F_off_B:      tag = "  (B)"
    else:                  tag = "  (C-PPR)"
    print(f"  {rank:2d}.  gain={imp[i]:>10.1f}  {all_feat_names[i]}{tag}")

Model                                                F1@0.5   F1@thr      AUC   PR-AUC
  RF C1 [233]                                        0.8149   0.7848   0.9209   0.8067
  LightGBM C1 [233]                                  0.8004   0.7842   0.9279   0.8041
  XGBoost  C1 [233]                                  0.6264   0.7845   0.9266   0.8034
  Ensemble RF+LGB+XGB (val weights)                  0.8062   0.8118   0.9274   0.8073

  Reference baselines (from earlier steps):
    A0 RF [108 intrinsic, no graph]       F1=0.8021  AUC=0.9026  PR-AUC=0.7855
    A1 RF [108 + 103 traj]                F1=0.8098  AUC=0.9196  PR-AUC=0.8029
    A3 RF [A1 + 17 Block B]               F1=0.8122  AUC=0.9160  PR-AUC=0.8027
    C1 RF [A3 + 5 PPR] (step 4)           F1=0.8149  AUC=0.9209  PR-AUC=0.8067

Per-timestep test F1 — best model (val-tuned threshold)
best by F1@thr: Ensemble RF+LGB+XGB (val weights)
  t=35  n=1341  illicit=182  F1=0.9669
  t=36  n=1708  illicit= 33  F1=0.9565
  t=37  n= 498  ill